In [1]:
import json, glob, os
from datetime import datetime, timezone

CADETS_FILES = sorted(glob.glob("../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json*"))

# April 6 attack window, UTC (11:21-12:08 EDT = 15:21-16:08 UTC); generous margin
WIN_START = int(datetime(2018, 4, 6, 14, 0, tzinfo=timezone.utc).timestamp() * 1e9)
WIN_END   = int(datetime(2018, 4, 6, 17, 0, tzinfo=timezone.utc).timestamp() * 1e9)

# IOCs from Ground Truth Report section 3.1 (April 6 CADETS Nginx backdoor)
IOC_IPS = {"81.49.200.166","78.205.235.65","200.36.109.214","139.123.0.113",
           "152.111.159.139","154.145.113.18","154.143.113.18","61.167.39.128"}
IOC_PATHS = {"/tmp/vUgefal","/var/log/devc"}

P = "com.bbn.tc.schema.avro.cdm18."

def cdm_type(datum):
    return next(iter(datum)).replace(P, "")

# ---- PASS 1: build UUID lookups ----
subjects, files, netflows = {}, {}, {}
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            t = cdm_type(d)
            obj = d[P + t]
            u = obj.get("uuid")
            if t == "Subject":
                props = obj.get("properties", {}).get("map", {})
                subjects[u] = props.get("exec") or obj.get("cmdLine") or "process"
            elif t == "FileObject":
                files[u] = obj.get("type", "FILE")
            elif t == "NetFlowObject":
                netflows[u] = f'{obj.get("remoteAddress")}:{obj.get("remotePort")}'

print("Subjects:", len(subjects), "Files:", len(files), "NetFlows:", len(netflows))

# ---- PASS 2: collect events in window ----
events = []
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try: d = json.loads(line)["datum"]
            except: continue
            if cdm_type(d) != "Event": continue
            e = d[P + "Event"]
            ts = e.get("timestampNanos", 0)
            if not (WIN_START <= ts <= WIN_END): continue
            events.append(e)

print("Events in window:", len(events))
if events:
    lo = min(e["timestampNanos"] for e in events)
    hi = max(e["timestampNanos"] for e in events)
    fmt = lambda ns: datetime.fromtimestamp(ns/1e9, timezone.utc).strftime("%H:%M:%S")
    print("Window actually spans:", fmt(lo), "to", fmt(hi), "UTC")

Subjects: 74187 Files: 845797 NetFlows: 26518
Events in window: 376379
Window actually spans: 14:00:00 to 16:08:59 UTC


In [2]:
from collections import Counter

types = Counter()
mal_types = Counter()
mal_hits = 0

def resolve(uuid, kind):
    if kind == "subj": return subjects.get(uuid)
    if kind == "file": return files.get(uuid)
    return None

for e in events:
    et = e.get("type", "?")
    types[et] += 1

    # gather any IP/path touched by this event
    touched_ips, touched_paths = set(), set()
    for k in ("predicateObject", "predicateObject2"):
        ref = e.get(k)
        if ref:
            u = ref.get(P + "UUID")
            nf = netflows.get(u)
            if nf: touched_ips.add(nf.split(":")[0])
    for k in ("predicateObjectPath", "predicateObject2Path"):
        pth = e.get(k)
        if pth:
            touched_paths.add(pth.get("string") if isinstance(pth, dict) else pth)

    is_mal = bool(touched_ips & IOC_IPS) or bool(touched_paths & IOC_PATHS)
    if is_mal:
        mal_hits += 1
        mal_types[et] += 1

print("Total event types in window:")
for t, c in types.most_common():
    print(f"  {t:30s} {c}")
print("\nMalicious events (IOC-matched):", mal_hits)
print("Malicious by type:")
for t, c in mal_types.most_common():
    print(f"  {t:30s} {c}")

Total event types in window:
  EVENT_READ                     110871
  EVENT_CLOSE                    49863
  EVENT_MMAP                     45882
  EVENT_FCNTL                    38396
  EVENT_WRITE                    36435
  EVENT_LSEEK                    36015
  EVENT_OPEN                     29481
  EVENT_CREATE_OBJECT            8855
  EVENT_RECVFROM                 2770
  EVENT_CHANGE_PRINCIPAL         2426
  EVENT_FORK                     2167
  EVENT_EXIT                     2107
  EVENT_EXECUTE                  1968
  EVENT_MODIFY_PROCESS           1868
  EVENT_CONNECT                  1823
  EVENT_SENDTO                   1331
  EVENT_MODIFY_FILE_ATTRIBUTES   1295
  EVENT_UNLINK                   845
  EVENT_ACCEPT                   572
  EVENT_ADD_OBJECT_ATTRIBUTE     288
  EVENT_MPROTECT                 247
  EVENT_RENAME                   214
  EVENT_LINK                     113
  EVENT_LOGIN                    112
  EVENT_RECVMSG                  108
  EVENT_TRUNCATE     

In [3]:
import random
random.seed(42)

# event types that carry incident meaning
SEMANTIC = {"EVENT_EXECUTE","EVENT_WRITE","EVENT_CREATE_OBJECT","EVENT_FORK",
            "EVENT_MODIFY_FILE_ATTRIBUTES","EVENT_MODIFY_PROCESS","EVENT_UNLINK",
            "EVENT_CHANGE_PRINCIPAL","EVENT_RENAME","EVENT_LINK","EVENT_LOGIN",
            "EVENT_MPROTECT","EVENT_TRUNCATE"}
TRAFFIC  = {"EVENT_CONNECT","EVENT_SENDTO","EVENT_RECVFROM","EVENT_ACCEPT",
            "EVENT_SENDMSG","EVENT_RECVMSG","EVENT_BIND"}

def event_targets(e):
    ips, paths = set(), set()
    for k in ("predicateObject","predicateObject2"):
        ref = e.get(k)
        if ref:
            u = ref.get(P+"UUID")
            nf = netflows.get(u)
            if nf: ips.add(nf.split(":")[0])
    for k in ("predicateObjectPath","predicateObject2Path"):
        pth = e.get(k)
        if pth:
            s = pth.get("string") if isinstance(pth, dict) else pth
            if s: paths.add(s)
    return ips, paths

def is_malicious(ips, paths):
    return bool(ips & IOC_IPS) or any(any(ioc in p for ioc in IOC_PATHS) for p in paths)

# ---- classify + collapse ----
semantic_events, traffic_channels = [], {}
for e in events:
    et = e.get("type","?")
    ips, paths = event_targets(e)
    mal = is_malicious(ips, paths)
    subj_ref = e.get("subject") or {}
    eprops = (e.get("properties") or {}).get("map") or {}
    proc = eprops.get("exec") or subjects.get(subj_ref.get(P+"UUID")) or "process"

    if et in TRAFFIC:
        ip = next(iter(ips), None)
        if ip:
            key = (proc, ip)
            if key not in traffic_channels:
                traffic_channels[key] = {"proc":proc,"ip":ip,"count":0,"mal":mal,"ts":e.get("timestampNanos")}
            traffic_channels[key]["count"] += 1
    elif et in SEMANTIC:
        tgt = next(iter(paths), None) or next(iter(ips), None) or "a socket/pipe"
        semantic_events.append({"type":et,"proc":proc,"target":tgt,
                                "mal":mal,"ts":e.get("timestampNanos")})

# ---- split malicious / benign ----
mal_sem = [x for x in semantic_events if x["mal"]]
ben_sem = [x for x in semantic_events if not x["mal"]]
mal_chan = [v for v in traffic_channels.values() if v["mal"]]
ben_chan = [v for v in traffic_channels.values() if not v["mal"]]

n_mal = len(mal_sem) + len(mal_chan)
ben_keep = random.sample(ben_sem, min(len(ben_sem), n_mal*10))
ben_chan_keep = random.sample(ben_chan, min(len(ben_chan), n_mal))  # connects are sparse, cap lightly

print(f"Malicious: {len(mal_sem)} semantic + {len(mal_chan)} channels = {n_mal}")
print(f"Benign kept: {len(ben_keep)} semantic + {len(ben_chan_keep)} channels")

# ---- render alert sentences ----
VERB = {"EVENT_EXECUTE":"executed","EVENT_WRITE":"wrote to","EVENT_CREATE_OBJECT":"created",
        "EVENT_FORK":"forked","EVENT_MODIFY_FILE_ATTRIBUTES":"changed permissions on",
        "EVENT_MODIFY_PROCESS":"modified process","EVENT_UNLINK":"deleted",
        "EVENT_CHANGE_PRINCIPAL":"changed privilege via","EVENT_RENAME":"renamed",
        "EVENT_LINK":"linked","EVENT_LOGIN":"logged in via","EVENT_MPROTECT":"changed memory protection on",
        "EVENT_TRUNCATE":"truncated"}

alerts = []
for x in mal_sem + ben_keep:
    alerts.append({"text":f'Process {x["proc"]} {VERB.get(x["type"],x["type"])} {x["target"]}',
                   "label":"malicious" if x["mal"] else "benign","ts":x["ts"]})
for c in mal_chan + ben_chan_keep:
    alerts.append({"text":f'Process {c["proc"]} connected to {c["ip"]} ({c["count"]} packets)',
                   "label":"malicious" if c["mal"] else "benign","ts":c["ts"]})

alerts.sort(key=lambda a:a["ts"])
print(f"\nTotal alerts: {len(alerts)}")
print("Sample malicious alerts:")
for a in [a for a in alerts if a["label"]=="malicious"][:12]:
    print("  ", a["text"])

Malicious: 8 semantic + 5 channels = 13
Benign kept: 130 semantic + 13 channels

Total alerts: 156
Sample malicious alerts:
   Process nginx connected to 81.49.200.166 (6 packets)
   Process nginx connected to 78.205.235.65 (302 packets)
   Process nginx wrote to <unknown>
   Process nginx wrote to <unknown>
   Process nginx connected to 200.36.109.214 (89 packets)
   Process nginx wrote to /tmp/vUgefal
   Process nginx changed permissions on /tmp/vUgefal
   Process master executed /tmp/vUgefal
   Process vUgefal connected to 139.123.0.113 (157 packets)
   Process nginx deleted /tmp/vUgefal
   Process vUgefal connected to 61.167.39.128 (2 packets)
   Process vUgefal wrote to /var/log/devc


In [4]:
from sentence_transformers import SentenceTransformer, util

texts = [a["text"] for a in alerts]
labels = [a["label"] for a in alerts]

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(texts, convert_to_tensor=True, show_progress_bar=True)

# same primitive as v6; threshold/min_size tuned for this small slice
communities = util.community_detection(emb, threshold=0.50, min_community_size=3)

print(f"{len(communities)} communities found\n")
for i, comm in enumerate(communities):
    comm_labels = [labels[j] for j in comm]
    n_mal = comm_labels.count("malicious")
    purity = max(comm_labels.count("malicious"), comm_labels.count("benign")) / len(comm)
    dom = "MALICIOUS" if n_mal > len(comm)/2 else "benign"
    print(f"Community {i}: {len(comm)} alerts | {n_mal} malicious | purity {purity:.2f} | dominant: {dom}")
    for j in comm[:6]:
        mark = "🔴" if labels[j]=="malicious" else "  "
        print(f"   {mark} {texts[j]}")
    if len(comm) > 6:
        print(f"   ... +{len(comm)-6} more")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

15 communities found

Community 0: 49 alerts | 0 malicious | purity 1.00 | dominant: benign
      Process wget wrote to /dev/tty
      Process wget wrote to /dev/tty
      Process wget wrote to /dev/tty
      Process wget wrote to /dev/tty
      Process wget wrote to /dev/tty
      Process wget wrote to /dev/tty
   ... +43 more

Community 1: 32 alerts | 0 malicious | purity 1.00 | dominant: benign
      Process master forked a socket/pipe
      Process sshd forked a socket/pipe
      Process sshd forked a socket/pipe
      Process sshd forked a socket/pipe
      Process rm modified process a socket/pipe
      Process sshd created a socket/pipe
   ... +26 more

Community 2: 10 alerts | 2 malicious | purity 0.80 | dominant: benign
      Process wget connected to 212.60.66.243 (3 packets)
   🔴 Process nginx wrote to <unknown>
   🔴 Process nginx wrote to <unknown>
      Process wget wrote to www.hbo.com/zone/index.html@ntrack_para1=leftnav_other0_4
      Process wget created /usr/home/user

In [5]:
def summarize(threshold, min_size=3):
    comms = util.community_detection(emb, threshold=threshold, min_community_size=min_size)
    mal_doms, purities, covered = 0, [], set()
    for comm in comms:
        cl = [labels[j] for j in comm]
        n_mal = cl.count("malicious")
        purities.append(max(n_mal, len(comm)-n_mal)/len(comm))
        if n_mal > len(comm)/2:
            mal_doms += 1
            for j in comm:
                if labels[j]=="malicious": covered.add(j)
    total_mal = sum(1 for l in labels if l=="malicious")
    avg_pur = sum(purities)/len(purities) if purities else 0
    return len(comms), mal_doms, len(covered), total_mal, avg_pur

print(f"{'thresh':>7} {'#comm':>6} {'#mal-dom':>9} {'mal-covered':>12} {'avg-purity':>11}")
for t in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
    n, md, cov, tot, ap = summarize(t)
    print(f"{t:>7.2f} {n:>6} {md:>9} {cov:>3}/{tot:<8} {ap:>11.2f}")

 thresh  #comm  #mal-dom  mal-covered  avg-purity
   0.45     12         1   4/13              0.89
   0.50     15         1   6/13              0.91
   0.55     13         3   9/13              0.90
   0.60     12         2   8/13              0.95
   0.65     15         2   8/13              0.98
   0.70     13         2   7/13              0.98
   0.75     14         2   6/13              0.98


In [6]:
import json, torch

# save stage-2 state including embeddings so reload is fully self-sufficient
state = {
    "threshold": 0.50,
    "alerts": alerts,
    "communities": [[int(j) for j in comm] for comm in communities],
}
with open("../data/darpa/cadets_apr6_stage2.json", "w") as f:
    json.dump(state, f)
torch.save(emb, "../data/darpa/cadets_apr6_emb.pt")

print("Saved:", len(alerts), "alerts,", len(communities), "communities + embeddings")

Saved: 156 alerts, 15 communities + embeddings


In [7]:
import json, torch

state = json.load(open("../data/darpa/cadets_apr6_stage2.json"))
alerts = state["alerts"]
communities = state["communities"]
labels = [a["label"] for a in alerts]
texts = [a["text"] for a in alerts]
emb = torch.load("../data/darpa/cadets_apr6_emb.pt")

# confirm what was loaded — sanity check, not decoration
print(f"Loaded {len(alerts)} alerts, {len(communities)} communities")
for i, comm in enumerate(communities):
    n_mal = sum(1 for j in comm if labels[j] == "malicious")
    if n_mal > len(comm) / 2:
        print(f"  Community {i} MALICIOUS-dominant ({n_mal}/{len(comm)}):")
        for j in comm:
            if labels[j] == "malicious":
                print(f"      {texts[j]}")

Loaded 156 alerts, 15 communities
  Community 8 MALICIOUS-dominant (6/6):
      Process vUgefal changed permissions on /var/log/devc
      Process nginx wrote to /tmp/vUgefal
      Process vUgefal connected to 61.167.39.128 (2 packets)
      Process vUgefal connected to 139.123.0.113 (157 packets)
      Process nginx changed permissions on /tmp/vUgefal
      Process nginx deleted /tmp/vUgefal


/tmp/ipykernel_175403/2799985756.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb = torch.load("../data/darpa/cadets_apr6_emb.pt")


In [8]:
import json, re
from collections import Counter
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH  = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"   # adjust path if different
RANDOM_SEED = 42

llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=-1, seed=RANDOM_SEED, verbose=False)
print("Model loaded.")

def normalise_entity(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9_]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text[:80]

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded.


In [9]:
SECURITY_RELATIONS = [
    'PERFORMS_RECONNAISSANCE','PERFORMS_PORT_SCAN','BRUTE_FORCES_CREDENTIAL',
    'ACCESS_CREDENTIALS','EXPLOITS_VULNERABILITY','ESTABLISHES_C2',
    'PERFORMS_BEACONING','CAUSES_DENIAL_OF_SERVICE','MOVES_LATERALLY',
    'EXFILTRATES_DATA','EXECUTES_PAYLOAD',
]
HOST_RELATIONS = SECURITY_RELATIONS + [
    'CHANGES_PERMISSIONS','DELETES_FILE','INJECTS_PROCESS','WRITES_FILE',
]

RELATION_TO_TECHNIQUES = {
    'PERFORMS_RECONNAISSANCE':  ['T1046','T1595','T1590'],
    'PERFORMS_PORT_SCAN':       ['T1046','T1595'],
    'BRUTE_FORCES_CREDENTIAL':  ['T1110','T1110.001','T1110.003'],
    'ACCESS_CREDENTIALS':       ['T1555','T1078','T1110'],
    'EXPLOITS_VULNERABILITY':   ['T1190','T1203'],
    'ESTABLISHES_C2':           ['T1071','T1071.001','T1071.004'],
    'PERFORMS_BEACONING':       ['T1071','T1071.004'],
    'CAUSES_DENIAL_OF_SERVICE': ['T1498','T1499','T1499.001'],
    'MOVES_LATERALLY':          ['T1021','T1570'],
    'EXFILTRATES_DATA':         ['T1041','T1048'],
    'EXECUTES_PAYLOAD':         ['T1059','T1059.007','T1105'],
    # host extensions
    'CHANGES_PERMISSIONS':      ['T1222','T1548'],
    'DELETES_FILE':             ['T1070.004','T1070'],
    'INJECTS_PROCESS':          ['T1055','T1055.001'],
    'WRITES_FILE':              ['T1105'],
}

TRIPLE_SCHEMA = {
    'type':'object','properties':{'triples':{'type':'array','minItems':1,'maxItems':4,
    'items':{'type':'object','properties':{
        'subject':{'type':'string'},
        'relation':{'type':'string','enum':HOST_RELATIONS},
        'target':{'type':'string'}},
    'required':['subject','relation','target']}}},'required':['triples']}
host_grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))

HOST_RELATION_GUIDE = """
Relation definitions (use exactly as written):
  EXPLOITS_VULNERABILITY  : initial exploit of a service (e.g. malformed request hitting a web server)
  ESTABLISHES_C2          : outbound connection to an external command-and-control address
  EXECUTES_PAYLOAD        : running a dropped binary or command on the host
  WRITES_FILE             : writing/dropping a file to disk (e.g. an implant payload)
  CHANGES_PERMISSIONS     : changing file permissions, often to enable execution or elevation
  INJECTS_PROCESS         : injecting code into another running process
  DELETES_FILE            : removing a file, often to cover tracks
  PERFORMS_PORT_SCAN      : probing multiple ports/hosts on the internal network
  EXFILTRATES_DATA        : transferring files or data off the host
""".strip()

def extract_triples_host(alert_texts, community_id):
    block = '\n'.join(f'- {t}' for t in alert_texts[:6])
    assert 'malicious' not in block and 'benign' not in block, f"Label leakage in {community_id}"
    prompt = f"""[INST] You are a cybersecurity analyst analyzing host audit events.
Read the host alerts below and extract between 1 and 4 semantic triples describing
the attack behaviour using standard security terminology.

{HOST_RELATION_GUIDE}

For each triple:
- subject : the process or actor observed (e.g. 'nginx', 'dropped_implant')
- relation: choose the ONE relation that best fits the observed action
- target  : the specific file, address, or process acted upon

Name what you observe. No generic placeholders.

Example (web exploit with dropped payload):
{{"triples": [
  {{"subject": "nginx",          "relation": "EXPLOITS_VULNERABILITY", "target": "web_server_process"}},
  {{"subject": "nginx",          "relation": "ESTABLISHES_C2",         "target": "external_c2_address"}},
  {{"subject": "nginx",          "relation": "WRITES_FILE",            "target": "tmp_implant_binary"}},
  {{"subject": "dropped_binary", "relation": "EXECUTES_PAYLOAD",       "target": "root_shell"}}
]}}

ALERTS (Community {community_id}):
{block}

Return ONLY valid JSON. [/INST]"""
    out = llm(prompt, max_tokens=512, temperature=0, seed=RANDOM_SEED,
              grammar=host_grammar, repeat_penalty=1.1, stop=['[/INST]'])
    raw = out['choices'][0]['text'].strip()
    try:
        triples = json.loads(raw).get('triples', [])
    except Exception as e:
        print(f'  [{community_id}] parse failed: {e}'); triples = []
    valid = set(HOST_RELATIONS); validated = []
    for t in triples:
        subj = normalise_entity(str(t.get('subject',''))); rel = str(t.get('relation','')).upper().strip()
        tgt = normalise_entity(str(t.get('target','')))
        if not subj or not tgt: continue
        if rel not in valid: rel = 'EXECUTES_PAYLOAD'
        validated.append({'subject':subj,'relation':rel,'target':tgt})
    return validated

# find malicious-dominant communities by label, not by fixed index
def malicious_dominant(communities, labels):
    return [i for i, comm in enumerate(communities)
            if sum(1 for j in comm if labels[j] == "malicious") > len(comm) / 2]

MAL_COMMS = malicious_dominant(communities, labels)
print("Malicious-dominant communities:", MAL_COMMS, "\n")

community_triples_darpa = {}
for cid in MAL_COMMS:
    member_texts = [texts[j] for j in communities[cid]]
    tr = extract_triples_host(member_texts, cid)
    community_triples_darpa[str(cid)] = tr
    print(f"Community {cid}:")
    for t in tr:
        print(f"   ({t['subject']}, {t['relation']}, {t['target']})")
    print()

Malicious-dominant communities: [8] 

Community 8:
   (vugefal, WRITES_FILE, var_log_devc)
   (vugefal, CHANGES_PERMISSIONS, tmp_vugefal)
   (vugefal, EXFILTRATES_DATA, 61_167_39_128)
   (vugefal, ESTABLISHES_C2, 139_123_0_113)



In [10]:
import numpy as np
from sentence_transformers import util as sutil

# --- load ATT&CK techniques from raw STIX bundle (built fresh, no saved index) ---
def load_attck(path="../data/attck/enterprise-attack.json"):
    bundle = json.load(open(path))
    techs = {}
    for obj in bundle["objects"]:
        if obj.get("type") != "attack-pattern" or obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue
        tid = next((r["external_id"] for r in obj.get("external_references", [])
                    if r.get("source_name") == "mitre-attack"), None)
        if not tid:
            continue
        tactics = [p["phase_name"] for p in obj.get("kill_chain_phases", [])
                   if p.get("kill_chain_name") == "mitre-attack"]
        techs[tid] = {
            "name": obj.get("name", ""),
            "description": obj.get("description", ""),
            "tactic": tactics[0] if tactics else "",
        }
    return techs

attck = load_attck()
tech_ids = list(attck.keys())
tech_texts = [f"{attck[t]['name']}. {attck[t]['description']}" for t in tech_ids]
tech_embs = model.encode(tech_texts, convert_to_tensor=True, show_progress_bar=True)
print(f"Loaded + embedded {len(tech_ids)} ATT&CK techniques")

TOP_K = 5

# --- ground truth from report, keyed by content (robust to renumbering) ---
def assign_gt(triples):
    rels = [t["relation"] for t in triples]
    if any(r in ("WRITES_FILE","CHANGES_PERMISSIONS","DELETES_FILE") for r in rels):
        return "T1105"   # ingress tool transfer / staging
    if any(r in ("EXPLOITS_VULNERABILITY","ESTABLISHES_C2","PERFORMS_PORT_SCAN") for r in rels):
        return "T1190"   # exploit public-facing application
    return None

def parent_match(gt, pred):
    if not gt or not pred or pred == "Unknown": return False
    return gt.split(".")[0] == pred.split(".")[0]

# --- retrieval: cosine search over in-memory technique embeddings ---
def retrieve(query, k=TOP_K):
    q = model.encode(query, convert_to_tensor=True)
    sims = sutil.pytorch_cos_sim(q, tech_embs)[0].cpu().numpy()
    top = np.argsort(-sims)[:k]
    return [tech_ids[i] for i in top]

# --- KG-anchored rerank: candidates from dominant relation, ranked by alert-text sim ---
def kg_candidates(triples):
    rels = [t["relation"] for t in triples]
    if not rels: return []
    dom = Counter(rels).most_common(1)[0][0]
    return [t for t in RELATION_TO_TECHNIQUES.get(dom, []) if t in attck]

results = []
for cid in MAL_COMMS:
    triples = community_triples_darpa[str(cid)]
    gt = assign_gt(triples)
    alert_text = " ".join(texts[j] for j in communities[cid])
    triple_text = " ".join(f"{t['subject']} {t['relation']} {t['target']}" for t in triples)
    cands = kg_candidates(triples)

    retrieved_ids = retrieve(alert_text[:400] + " " + triple_text)

    # rerank within KG candidate set; fall back to plain retrieval if none
    if cands:
        q_emb = model.encode(alert_text[:400], convert_to_tensor=True)
        c_embs = tech_embs[[tech_ids.index(t) for t in cands]]
        sims = sutil.pytorch_cos_sim(q_emb, c_embs)[0].cpu().tolist()
        pred = cands[int(np.argmax(sims))]
    else:
        pred = retrieved_ids[0] if retrieved_ids else "Unknown"

    results.append({
        "cid": cid, "gt": gt, "pred": pred,
        "dom_rel": Counter(t["relation"] for t in triples).most_common(1)[0][0] if triples else "none",
        "retrieval_hit": gt in retrieved_ids,
        "parent_match": parent_match(gt, pred),
    })

print(f"\n{'cid':>4} {'GT':>8} {'pred':>8} {'dom_rel':>22} {'retr_hit':>9} {'parent':>7}")
for r in results:
    print(f"{r['cid']:>4} {r['gt']:>8} {r['pred']:>8} {r['dom_rel']:>22} "
          f"{str(r['retrieval_hit']):>9} {str(r['parent_match']):>7}")

hits = sum(r["parent_match"] for r in results)
print(f"\nParent-match: {hits}/{len(results)} = {hits/len(results):.1%}")

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Loaded + embedded 691 ATT&CK techniques

 cid       GT     pred                dom_rel  retr_hit  parent
   8    T1105    T1105            WRITES_FILE     False    True

Parent-match: 1/1 = 100.0%
